In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

In [2]:
# ============================================================
# 2. LOAD DATA
# ============================================================

matches = pd.read_csv("matches.csv")

events = pd.read_csv("events.csv")

team_match_stats = pd.read_csv(
    "team_match_stats.csv"
)

positions = pd.read_csv(
    "positions.csv"
)

print(team_match_stats.shape)

team_match_stats.head()

/var/folders/l1/c11rhp6x6pl9nqfm7t9hdxmw0000gn/T/ipykernel_43131/160511178.py:7: DtypeWarning: Columns (0,1,2,3,4,5,12,14,15,16,19,20,22,27,32,37,40,42,43,44,45,46,47,48,49,51,52,53,55,56,57,58,59,76,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118) have mixed types. Specify dtype option on import or set low_memory=False.
  events = pd.read_csv("events.csv")


(342, 19)


,competition_name,season_name,competition_short,tournament_id,match_id,team,total_events,shots,passes,carries,pressures,recoveries,duels,interceptions,blocks,clearances,xg,completed_passes,pass_completion_pct
0,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Côte d'Ivoire,1998,10,598,510,103,48,20,9,12,9,0.706896,517,86.454849
1,African Cup of Nations,2023,AFCON,AFCON_2023,3920384,Guinea-Bissau,1450,10,394,301,145,36,26,15,16,17,0.529615,328,83.248731
2,African Cup of Nations,2023,AFCON,AFCON_2023,3920385,Egypt,2025,22,589,497,141,59,25,11,9,12,1.929200,512,86.926995
3,African Cup of Nations,2023,AFCON,AFCON_2023,3920385,Mozambique,1445,6,373,329,140,31,19,11,20,40,0.728199,309,82.841823
4,African Cup of Nations,2023,AFCON,AFCON_2023,3920387,Cape Verde Islands,1511,15,417,320,122,34,24,9,21,17,2.720214,341,81.774580


In [3]:
# ============================================================
# 3. CREATE TEAM MASTER BASE
# ============================================================

team_master = (
    team_match_stats
    .groupby([
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team"
    ])
    .agg(
        matches=("match_id", "nunique"),

        shots=("shots", "sum"),

        passes=("passes", "sum"),

        carries=("carries", "sum"),

        pressures=("pressures", "sum"),

        recoveries=("recoveries", "sum"),

        duels=("duels", "sum"),

        interceptions=("interceptions", "sum"),

        blocks=("blocks", "sum"),

        clearances=("clearances", "sum"),

        xg=("xg", "sum"),

        avg_pass_completion_pct=(
            "pass_completion_pct",
            "mean"
        )
    )
    .reset_index()
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,recoveries,duels,interceptions,blocks,clearances,xg,avg_pass_completion_pct
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,123,84,23,35,34,3.881383,84.272762
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,39,26,13,22,26,1.197465,72.297297
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,47,31,7,10,12,1.258992,71.804511
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,43,21,5,25,18,0.248547,78.066038
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,155,86,26,72,84,8.554618,80.278538


In [4]:
# ============================================================
# 4. ADD REAL GOALS
# ============================================================

home_goals = matches[
    [
        "competition_name",
        "season_name",
        "home_team",
        "home_score"
    ]
].rename(
    columns={
        "home_team": "team",
        "home_score": "goals"
    }
)

away_goals = matches[
    [
        "competition_name",
        "season_name",
        "away_team",
        "away_score"
    ]
].rename(
    columns={
        "away_team": "team",
        "away_score": "goals"
    }
)

all_goals = pd.concat(
    [home_goals, away_goals],
    ignore_index=True
)

team_goals = (
    all_goals
    .groupby([
        "competition_name",
        "season_name",
        "team"
    ])
    .agg(
        goals=("goals", "sum")
    )
    .reset_index()
)

team_master = team_master.merge(
    team_goals,
    on=[
        "competition_name",
        "season_name",
        "team"
    ],
    how="left"
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,recoveries,duels,interceptions,blocks,clearances,xg,avg_pass_completion_pct,goals
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,123,84,23,35,34,3.881383,84.272762,3
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,39,26,13,22,26,1.197465,72.297297,1
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,47,31,7,10,12,1.258992,71.804511,2
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,43,21,5,25,18,0.248547,78.066038,1
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,155,86,26,72,84,8.554618,80.278538,4


In [5]:
# ============================================================
# ADD TEAM RESULTS SUMMARY
# ============================================================

home_results = matches[
    [
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_result"
    ]
].copy()

home_results = home_results.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "goals_for",
        "away_score": "goals_against",
        "home_result": "result"
    }
)

away_results = matches[
    [
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "away_team",
        "home_team",
        "away_score",
        "home_score",
        "away_result"
    ]
].copy()

away_results = away_results.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "goals_for",
        "home_score": "goals_against",
        "away_result": "result"
    }
)

team_results_long = pd.concat(
    [home_results, away_results],
    ignore_index=True
)

team_results_summary = (
    team_results_long
    .groupby([
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team"
    ])
    .agg(
        wins=("result", lambda x: (x == "Win").sum()),
        draws=("result", lambda x: (x == "Draw").sum()),
        losses=("result", lambda x: (x == "Loss").sum()),
        goals_for=("goals_for", "sum"),
        goals_against=("goals_against", "sum")
    )
    .reset_index()
)

team_results_summary["goal_difference"] = (
    team_results_summary["goals_for"] -
    team_results_summary["goals_against"]
)

team_master = team_master.merge(
    team_results_summary,
    on=[
        "competition_name",
        "season_name",
        "competition_short",
        "tournament_id",
        "team"
    ],
    how="left"
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,clearances,xg,avg_pass_completion_pct,goals,wins,draws,losses,goals_for,goals_against,goal_difference
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,34,3.881383,84.272762,3,0,2,1,3,4,-1
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,26,1.197465,72.297297,1,0,1,0,1,1,0
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,12,1.258992,71.804511,2,0,1,0,2,2,0
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,18,0.248547,78.066038,1,0,0,1,1,3,-2
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,84,8.554618,80.278538,4,1,2,0,4,3,1


In [6]:
# ============================================================
# 5. CREATE PER MATCH METRICS
# ============================================================

volume_cols = [
    "shots",
    "passes",
    "carries",
    "pressures",
    "recoveries",
    "duels",
    "interceptions",
    "blocks",
    "clearances",
    "xg",
    "goals"
]

for col in volume_cols:
    team_master[f"{col}_per_match"] = (
        team_master[col] / team_master["matches"]
    )

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,passes_per_match,carries_per_match,pressures_per_match,recoveries_per_match,duels_per_match,interceptions_per_match,blocks_per_match,clearances_per_match,xg_per_match,goals_per_match
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,555.0,449.000000,103.666667,41.000000,28.000000,7.666667,11.666667,11.333333,1.293794,1.000000
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,296.0,218.000000,189.000000,39.000000,26.000000,13.000000,22.000000,26.000000,1.197465,1.000000
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,266.0,196.000000,120.000000,47.000000,31.000000,7.000000,10.000000,12.000000,1.258992,2.000000
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,424.0,332.000000,160.000000,43.000000,21.000000,5.000000,25.000000,18.000000,0.248547,1.000000
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,461.0,361.333333,151.333333,51.666667,28.666667,8.666667,24.000000,28.000000,2.851539,1.333333


In [7]:
# ============================================================
# 6. CREATE STYLE METRICS
# ============================================================

team_master["tempo"] = (
    team_master["passes_per_match"] +
    team_master["carries_per_match"]
)

team_master["directness"] = np.where(
    team_master["passes"] > 0,
    team_master["shots"] / team_master["passes"] * 100,
    0
)

team_master["pressing_intensity"] = (
    team_master["pressures_per_match"] +
    team_master["recoveries_per_match"]
)

team_master["defensive_activity"] = (
    team_master["duels_per_match"] +
    team_master["interceptions_per_match"] +
    team_master["blocks_per_match"] +
    team_master["clearances_per_match"]
)

team_master["attacking_threat"] = (
    team_master["xg_per_match"] * 0.6 +
    team_master["shots_per_match"] * 0.4
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,interceptions_per_match,blocks_per_match,clearances_per_match,xg_per_match,goals_per_match,tempo,directness,pressing_intensity,defensive_activity,attacking_threat
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,7.666667,11.666667,11.333333,1.293794,1.000000,1004.000000,2.822823,144.666667,58.666667,7.042943
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,13.000000,22.000000,26.000000,1.197465,1.000000,514.000000,2.364865,228.000000,87.000000,3.518479
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,7.000000,10.000000,12.000000,1.258992,2.000000,462.000000,2.631579,167.000000,60.000000,3.555395
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,5.000000,25.000000,18.000000,0.248547,1.000000,756.000000,1.179245,203.000000,69.000000,2.149128
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,8.666667,24.000000,28.000000,2.851539,1.333333,822.333333,3.687636,203.000000,89.333333,8.510924


In [8]:
# ============================================================
# 7. CREATE TOURNAMENT PERCENTILES
# ============================================================

ranking_metrics = [
    "shots_per_match",
    "passes_per_match",
    "carries_per_match",
    "pressures_per_match",
    "recoveries_per_match",
    "xg_per_match",
    "goals_per_match",
    "avg_pass_completion_pct",
    "tempo",
    "directness",
    "pressing_intensity",
    "defensive_activity",
    "attacking_threat"
]

for metric in ranking_metrics:
    team_master[f"{metric}_rank"] = (
        team_master
        .groupby("tournament_id")[metric]
        .rank(pct=True)
    )

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,pressures_per_match_rank,recoveries_per_match_rank,xg_per_match_rank,goals_per_match_rank,avg_pass_completion_pct_rank,tempo_rank,directness_rank,pressing_intensity_rank,defensive_activity_rank,attacking_threat_rank
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,0.041667,0.354167,0.583333,0.479167,0.916667,1.000000,0.625000,0.041667,0.083333,0.833333
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,0.958333,0.187500,0.458333,0.479167,0.208333,0.187500,0.250000,0.916667,0.791667,0.208333
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,0.125000,0.791667,0.541667,0.895833,0.166667,0.125000,0.541667,0.208333,0.125000,0.250000
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,0.750000,0.645833,0.083333,0.479167,0.416667,0.583333,0.083333,0.770833,0.166667,0.083333
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,0.625000,0.958333,0.916667,0.729167,0.583333,0.666667,0.791667,0.770833,0.833333,0.958333


In [9]:
# ============================================================
# 8. CLASSIFY GENERAL TEAM STYLE
# ============================================================

def classify_general_style(row):

    if row["passes_per_match_rank"] >= 0.75 and row["avg_pass_completion_pct_rank"] >= 0.60:
        return "Possession / Control"

    elif row["directness_rank"] >= 0.75 and row["passes_per_match_rank"] <= 0.50:
        return "Direct / Vertical"

    elif row["pressing_intensity_rank"] >= 0.75:
        return "High Intensity / Pressing"

    elif row["attacking_threat_rank"] >= 0.75:
        return "Attacking Threat"

    else:
        return "Balanced / Mixed"

team_master["general_style"] = team_master.apply(
    classify_general_style,
    axis=1
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,recoveries_per_match_rank,xg_per_match_rank,goals_per_match_rank,avg_pass_completion_pct_rank,tempo_rank,directness_rank,pressing_intensity_rank,defensive_activity_rank,attacking_threat_rank,general_style
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,0.354167,0.583333,0.479167,0.916667,1.000000,0.625000,0.041667,0.083333,0.833333,Possession / Control
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,0.187500,0.458333,0.479167,0.208333,0.187500,0.250000,0.916667,0.791667,0.208333,High Intensity / Pressing
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,0.791667,0.541667,0.895833,0.166667,0.125000,0.541667,0.208333,0.125000,0.250000,Balanced / Mixed
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,0.645833,0.083333,0.479167,0.416667,0.583333,0.083333,0.770833,0.166667,0.083333,High Intensity / Pressing
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,0.958333,0.916667,0.729167,0.583333,0.666667,0.791667,0.770833,0.833333,0.958333,High Intensity / Pressing


In [10]:
# ============================================================
# 9. CLASSIFY OFFENSIVE AND DEFENSIVE STYLE
# ============================================================

def classify_offensive_style(row):

    if row["xg_per_match_rank"] >= 0.75 and row["shots_per_match_rank"] >= 0.75:
        return "High Chance Creation"

    elif row["passes_per_match_rank"] >= 0.75:
        return "Possession-Based Attack"

    elif row["directness_rank"] >= 0.75:
        return "Direct Attack"

    elif row["carries_per_match_rank"] >= 0.75:
        return "Carry-Based Progression"

    else:
        return "Balanced Attack"


def classify_defensive_style(row):

    if row["pressing_intensity_rank"] >= 0.75:
        return "Aggressive Pressing"

    elif row["defensive_activity_rank"] >= 0.75:
        return "High Defensive Activity"

    elif row["recoveries_per_match_rank"] >= 0.75:
        return "Recovery-Oriented"

    else:
        return "Balanced Defense"


team_master["offensive_style"] = team_master.apply(
    classify_offensive_style,
    axis=1
)

team_master["defensive_style"] = team_master.apply(
    classify_defensive_style,
    axis=1
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,goals_per_match_rank,avg_pass_completion_pct_rank,tempo_rank,directness_rank,pressing_intensity_rank,defensive_activity_rank,attacking_threat_rank,general_style,offensive_style,defensive_style
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,0.479167,0.916667,1.000000,0.625000,0.041667,0.083333,0.833333,Possession / Control,Possession-Based Attack,Balanced Defense
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,0.479167,0.208333,0.187500,0.250000,0.916667,0.791667,0.208333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,0.895833,0.166667,0.125000,0.541667,0.208333,0.125000,0.250000,Balanced / Mixed,Balanced Attack,Recovery-Oriented
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,0.479167,0.416667,0.583333,0.083333,0.770833,0.166667,0.083333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,0.729167,0.583333,0.666667,0.791667,0.770833,0.833333,0.958333,High Intensity / Pressing,High Chance Creation,Aggressive Pressing


In [11]:
# ============================================================
# 10. CREATE TEAM STRENGTHS AND WEAKNESSES
# ============================================================

def get_team_strengths(row):

    strengths = []

    if row["xg_per_match_rank"] >= 0.75:
        strengths.append("Strong chance generation")

    if row["shots_per_match_rank"] >= 0.75:
        strengths.append("High shot volume")

    if row["passes_per_match_rank"] >= 0.75:
        strengths.append("High possession volume")

    if row["avg_pass_completion_pct_rank"] >= 0.75:
        strengths.append("Strong passing accuracy")

    if row["pressing_intensity_rank"] >= 0.75:
        strengths.append("High pressing intensity")

    if row["recoveries_per_match_rank"] >= 0.75:
        strengths.append("Strong ball recovery activity")

    if row["directness_rank"] >= 0.75:
        strengths.append("Direct attacking profile")

    if len(strengths) == 0:
        strengths.append("Balanced profile without one dominant strength")

    return strengths


def get_team_weaknesses(row):

    weaknesses = []

    if row["xg_per_match_rank"] <= 0.25:
        weaknesses.append("Limited chance generation")

    if row["shots_per_match_rank"] <= 0.25:
        weaknesses.append("Low shot volume")

    if row["passes_per_match_rank"] <= 0.25:
        weaknesses.append("Low possession volume")

    if row["avg_pass_completion_pct_rank"] <= 0.25:
        weaknesses.append("Low passing accuracy")

    if row["pressing_intensity_rank"] <= 0.25:
        weaknesses.append("Low pressing activity")

    if row["recoveries_per_match_rank"] <= 0.25:
        weaknesses.append("Limited ball recovery activity")

    if len(weaknesses) == 0:
        weaknesses.append("No clear major weakness from selected indicators")

    return weaknesses


team_master["strengths"] = team_master.apply(
    get_team_strengths,
    axis=1
)

team_master["weaknesses"] = team_master.apply(
    get_team_weaknesses,
    axis=1
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,tempo_rank,directness_rank,pressing_intensity_rank,defensive_activity_rank,attacking_threat_rank,general_style,offensive_style,defensive_style,strengths,weaknesses
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,1.000000,0.625000,0.041667,0.083333,0.833333,Possession / Control,Possession-Based Attack,Balanced Defense,"[High shot volume, High possession volume, Str...",[Low pressing activity]
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,0.187500,0.250000,0.916667,0.791667,0.208333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Low shot volume, Low possession volume, Low p..."
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,0.125000,0.541667,0.208333,0.125000,0.250000,Balanced / Mixed,Balanced Attack,Recovery-Oriented,[Strong ball recovery activity],"[Low shot volume, Low possession volume, Low p..."
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,0.583333,0.083333,0.770833,0.166667,0.083333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Limited chance generation, Low shot volume]"
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,0.666667,0.791667,0.770833,0.833333,0.958333,High Intensity / Pressing,High Chance Creation,Aggressive Pressing,"[Strong chance generation, High shot volume, H...",[No clear major weakness from selected indicat...


In [12]:
# ============================================================
# 11. ADD MOST USED FORMATION
# ============================================================

formation_summary = (
    positions[
        positions["starter"] == True
    ]
    .groupby([
        "competition_name",
        "season_name",
        "team",
        "match_id"
    ])
    .agg(
        formation=("position", lambda x: "-".join(
            x.dropna().astype(str).tolist()
        ))
    )
    .reset_index()
)

# Better practical formation proxy: use lineup position count string
team_formations = (
    positions[
        positions["starter"] == True
    ]
    .groupby([
        "competition_name",
        "season_name",
        "team",
        "match_id"
    ])
    .agg(
        starters=("player", "nunique")
    )
    .reset_index()
)

# For now, we keep the most common raw formation field if it exists
if "formation" in positions.columns:

    most_used_formation = (
        positions
        .dropna(subset=["formation"])
        .drop_duplicates([
            "competition_name",
            "season_name",
            "team",
            "match_id",
            "formation"
        ])
        .groupby([
            "competition_name",
            "season_name",
            "team",
            "formation"
        ])
        .agg(
            formation_matches=("match_id", "nunique")
        )
        .reset_index()
    )

    most_used_formation["formation_rank"] = (
        most_used_formation
        .groupby([
            "competition_name",
            "season_name",
            "team"
        ])["formation_matches"]
        .rank(method="first", ascending=False)
    )

    most_used_formation = most_used_formation[
        most_used_formation["formation_rank"] == 1
    ][
        [
            "competition_name",
            "season_name",
            "team",
            "formation"
        ]
    ].rename(
        columns={"formation": "most_used_formation"}
    )

    team_master = team_master.merge(
        most_used_formation,
        on=[
            "competition_name",
            "season_name",
            "team"
        ],
        how="left"
    )

else:
    team_master["most_used_formation"] = np.nan

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,directness_rank,pressing_intensity_rank,defensive_activity_rank,attacking_threat_rank,general_style,offensive_style,defensive_style,strengths,weaknesses,most_used_formation
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,0.625000,0.041667,0.083333,0.833333,Possession / Control,Possession-Based Attack,Balanced Defense,"[High shot volume, High possession volume, Str...",[Low pressing activity],NaN
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,0.250000,0.916667,0.791667,0.208333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Low shot volume, Low possession volume, Low p...",NaN
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,0.541667,0.208333,0.125000,0.250000,Balanced / Mixed,Balanced Attack,Recovery-Oriented,[Strong ball recovery activity],"[Low shot volume, Low possession volume, Low p...",NaN
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,0.083333,0.770833,0.166667,0.083333,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Limited chance generation, Low shot volume]",NaN
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,0.791667,0.770833,0.833333,0.958333,High Intensity / Pressing,High Chance Creation,Aggressive Pressing,"[Strong chance generation, High shot volume, H...",[No clear major weakness from selected indicat...,NaN


In [13]:
# ============================================================
# 12. CREATE TEAM AREA SCORES
# ============================================================

team_master["attack_score"] = (
    team_master["xg_per_match_rank"] * 0.45 +
    team_master["shots_per_match_rank"] * 0.30 +
    team_master["goals_per_match_rank"] * 0.25
) * 100

team_master["build_up_score"] = (
    team_master["passes_per_match_rank"] * 0.40 +
    team_master["avg_pass_completion_pct_rank"] * 0.30 +
    team_master["carries_per_match_rank"] * 0.30
) * 100

team_master["defense_score"] = (
    team_master["pressing_intensity_rank"] * 0.35 +
    team_master["recoveries_per_match_rank"] * 0.30 +
    team_master["defensive_activity_rank"] * 0.35
) * 100

team_master["directness_score"] = (
    team_master["directness_rank"] * 100
)

team_master.head()

,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,general_style,offensive_style,defensive_style,strengths,weaknesses,most_used_formation,attack_score,build_up_score,defense_score,directness_score
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,Possession / Control,Possession-Based Attack,Balanced Defense,"[High shot volume, High possession volume, Str...",[Low pressing activity],NaN,64.479167,97.500000,15.000000,62.500000
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Low shot volume, Low possession volume, Low p...",NaN,38.854167,19.583333,65.416667,25.000000
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,Balanced / Mixed,Balanced Attack,Recovery-Oriented,[Strong ball recovery activity],"[Low shot volume, Low possession volume, Low p...",NaN,53.020833,13.750000,35.416667,54.166667
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Limited chance generation, Low shot volume]",NaN,18.229167,51.666667,52.187500,8.333333
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,High Intensity / Pressing,High Chance Creation,Aggressive Pressing,"[Strong chance generation, High shot volume, H...",[No clear major weakness from selected indicat...,NaN,87.604167,64.166667,84.895833,79.166667


In [14]:
# ============================================================
# 13. ROUND AND SAVE TEAM MASTER
# ============================================================

numeric_cols = team_master.select_dtypes(
    include=np.number
).columns

team_master[numeric_cols] = team_master[numeric_cols].round(2)

team_master.to_csv(
    "team_master.csv",
    index=False
)

print("team_master.csv saved successfully!")
print(team_master.shape)

team_master.head()

team_master.csv saved successfully!
(96, 63)


,competition_name,season_name,competition_short,tournament_id,team,matches,shots,passes,carries,pressures,...,general_style,offensive_style,defensive_style,strengths,weaknesses,most_used_formation,attack_score,build_up_score,defense_score,directness_score
0,African Cup of Nations,2023,AFCON,AFCON_2023,Algeria,3,47,1665,1347,311,...,Possession / Control,Possession-Based Attack,Balanced Defense,"[High shot volume, High possession volume, Str...",[Low pressing activity],NaN,64.48,97.50,15.00,62.50
1,African Cup of Nations,2023,AFCON,AFCON_2023,Angola,1,7,296,218,189,...,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Low shot volume, Low possession volume, Low p...",NaN,38.85,19.58,65.42,25.00
2,African Cup of Nations,2023,AFCON,AFCON_2023,Burkina Faso,1,7,266,196,120,...,Balanced / Mixed,Balanced Attack,Recovery-Oriented,[Strong ball recovery activity],"[Low shot volume, Low possession volume, Low p...",NaN,53.02,13.75,35.42,54.17
3,African Cup of Nations,2023,AFCON,AFCON_2023,Cameroon,1,5,424,332,160,...,High Intensity / Pressing,Balanced Attack,Aggressive Pressing,[High pressing intensity],"[Limited chance generation, Low shot volume]",NaN,18.23,51.67,52.19,8.33
4,African Cup of Nations,2023,AFCON,AFCON_2023,Cape Verde Islands,3,51,1383,1084,454,...,High Intensity / Pressing,High Chance Creation,Aggressive Pressing,"[Strong chance generation, High shot volume, H...",[No clear major weakness from selected indicat...,NaN,87.60,64.17,84.90,79.17
